In [4]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error
import pingouin as pg
import os
import pickle
from IPython.display import display
from pipeline import Pipeline


In [5]:
# def show_results_table(r_list, mae_list, icc_list, feature_name, chunk_label):
#     """
#     Plot a color-coded table of results.
#     - High R and ICC values: green
#     - Medium R and ICC values: yellow  
#     - Low R and ICC values: red
#     - For MAE: low values (good) are green, medium are yellow, high are red
#     """
#     # Create DataFrame
#     df = pd.DataFrame({
#         'Subject': [f'Subject {i+1}' for i in range(len(r_list))],
#         'R': r_list,
#         'MAE': mae_list,
#         'ICC': icc_list
#     })
#     df = df.set_index('Subject')
    
#     # Define color mapping functions with better contrast
#     def color_r_icc(val):
#         """Color for R and ICC values (higher is better)"""
#         if val >= 0.8:
#             return 'background-color: #2d7a3e; color: white; font-weight: bold'  # Dark green with white text
#         elif val >= 0.5:
#             return 'background-color: #f4a460; color: black; font-weight: bold'  # Dark orange with black text
#         else:
#             return 'background-color: #c41e3a; color: white; font-weight: bold'  # Dark red with white text
    
#     def color_mae(val):
#         """Color for MAE values (lower is better)"""
#         if val <= 5:
#             return 'background-color: #2d7a3e; color: white; font-weight: bold'  # Dark green with white text
#         elif val <= 15:
#             return 'background-color: #f4a460; color: black; font-weight: bold'  # Dark orange with black text
#         else:
#             return 'background-color: #c41e3a; color: white; font-weight: bold'  # Dark red with white text
    
#     # Build a title that identifies the feature (RMSSD/SDNN) and chunk type (Random/First)
#     title = f"Results for {feature_name} ({chunk_label})"

#     # Apply styling and add the title as a caption so it shows on the table itself
#     styled_df = df.style.map(lambda x: color_r_icc(x), subset=['R', 'ICC']) \
#                         .map(lambda x: color_mae(x), subset=['MAE']) \
#                         .format({'R': '{:.4f}', 'MAE': '{:.4f}', 'ICC': '{:.4f}'}) \
#                         .set_caption(title) \
#                         .set_table_styles([{
#                             'selector': 'caption',
#                             'props': [('font-size', '16px'),
#                                       ('font-weight', 'bold'),
#                                       ('text-align', 'left'),
#                                       ('padding-bottom', '8px')]
#                         }])
    
#     # Display the styled table only (no duplicate text version)
#     display(styled_df)
#     return None


## Comparison between 30 seconds and 5 minutes (Random chuncks)

In [6]:
pipeline_stress_5m_30s_R = Pipeline(1, 300,30, chunk_method="random")

res_5m_30s_R = pipeline_stress_5m_30s_R.run_pipeline("G:/Master/Thesis/FLT/Code/WESAD")
# show_results_table(res_5m_30s_R['r_rmssd'], res_5m_30s_R['mae_rmssd'], res_5m_30s_R['icc_rmssd'], "RMSSD", "Random chunks")

# show_results_table(res_5m_30s_R['r_sdnn'], res_5m_30s_R['mae_sdnn'], res_5m_30s_R['icc_sdnn'], "SDNN", "Random chunks")


Processing file: S11.pkl in G:/Master/Thesis/FLT/Code/WESAD\data
Processing file: S2.pkl in G:/Master/Thesis/FLT/Code/WESAD\data


,R,MAE,ICC
Subject,,,
Subject 1,0.6151,7.7031,0.2847
Subject 2,-0.7571,15.6474,-0.5754


,R,MAE,ICC
Subject,,,
Subject 1,0.3600,16.4338,0.2322
Subject 2,0.1295,27.2971,0.0201


## Comparison between 30 seconds and 5 minutes (First 30 seconds chuncks)

In [7]:
pipeline_stress_5m_30s_F = Pipeline(1, 300,30, chunk_method="first")

res_5m_30s_F = pipeline_stress_5m_30s_F.run_pipeline("G:/Master\Thesis\FLT\Code\WESAD")

# show_results_table(res_5m_30s_F['r_rmssd'], res_5m_30s_F['mae_rmssd'], res_5m_30s_F['icc_rmssd'], "RMSSD", "First 30 seconds chunks")


<>:3: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
<>:3: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
C:\Users\User\AppData\Local\Temp\ipykernel_18128\323928766.py:3: SyntaxWarning: "\T" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\T"? A raw string is also an option.
  res_5m_30s_F = pipeline_stress_5m_30s_F.run_pipeline("G:/Master\Thesis\FLT\Code\WESAD")


Processing file: S11.pkl in G:/Master\Thesis\FLT\Code\WESAD\data
Processing file: S2.pkl in G:/Master\Thesis\FLT\Code\WESAD\data


,R,MAE,ICC
Subject,,,
Subject 1,-0.0696,7.6775,-0.0357
Subject 2,0.8398,3.2797,0.6429


,R,MAE,ICC
Subject,,,
Subject 1,0.7544,10.7525,0.5720
Subject 2,-0.0921,11.6762,-0.0914


## Comparison between 30 seconds and 5 minutes (Monte Carlo Simulation)

In [8]:
# Monte Carlo simulation of the random-chunk method.
# Run the random selection N_RUNS times and average the per-subject metrics.
N_RUNS = 1000
folder_path = "G:/Master/Thesis/FLT/Code/WESAD"

# Pipeline configured with the random chunk method (same settings as above)
pipeline_mc = Pipeline(1, 300, 30, chunk_method="random")

# --- Prepare the big chunks ONCE (they do not change between runs) ---
ecgs_list, labels_list = pipeline_mc.read_wesad(folder_path)
ecg_by_label = pipeline_mc.get_ecg_by_label(ecgs_list, labels_list, pipeline_mc.label)
ecg_big_chunks = pipeline_mc.chunk_ecg(ecg_by_label, time_in_sec=pipeline_mc.big_chunck_size)

# Big-chunk HRV is constant across runs, so compute it only once
rmssd_big, sdnn_big = pipeline_mc.get_hrv_parameters(ecg_big_chunks)

# --- Repeat the random selection N_RUNS times ---
r_rmssd_runs, mae_rmssd_runs, icc_rmssd_runs = [], [], []
r_sdnn_runs, mae_sdnn_runs, icc_sdnn_runs = [], [], []

for run in range(N_RUNS):
    ecg_small_chunks = pipeline_mc.random_chunk(ecg_big_chunks, time_in_sec=pipeline_mc.small_chunck_size)
    rmssd_small, sdnn_small = pipeline_mc.get_hrv_parameters(ecg_small_chunks)

    r_rmssd, mae_rmssd, icc_rmssd = pipeline_mc.get_statistics(rmssd_big, rmssd_small)
    r_sdnn, mae_sdnn, icc_sdnn = pipeline_mc.get_statistics(sdnn_big, sdnn_small)

    r_rmssd_runs.append(r_rmssd)
    mae_rmssd_runs.append(mae_rmssd)
    icc_rmssd_runs.append(icc_rmssd)
    r_sdnn_runs.append(r_sdnn)
    mae_sdnn_runs.append(mae_sdnn)
    icc_sdnn_runs.append(icc_sdnn)

    if (run + 1) % 100 == 0:
        print(f"Completed {run + 1}/{N_RUNS} runs")

# --- Average across runs (axis=0 averages over runs, keeping per-subject values) ---
avg_r_rmssd = np.mean(r_rmssd_runs, axis=0).tolist()
avg_mae_rmssd = np.mean(mae_rmssd_runs, axis=0).tolist()
avg_icc_rmssd = np.mean(icc_rmssd_runs, axis=0).tolist()
avg_r_sdnn = np.mean(r_sdnn_runs, axis=0).tolist()
avg_mae_sdnn = np.mean(mae_sdnn_runs, axis=0).tolist()
avg_icc_sdnn = np.mean(icc_sdnn_runs, axis=0).tolist()

# --- Show the averaged results in the table ---
# show_results_table(avg_r_rmssd, avg_mae_rmssd, avg_icc_rmssd, "RMSSD", f"Monte Carlo avg of {N_RUNS} runs")
# show_results_table(avg_r_sdnn, avg_mae_sdnn, avg_icc_sdnn, "SDNN", f"Monte Carlo avg of {N_RUNS} runs")

Processing file: S11.pkl in G:/Master/Thesis/FLT/Code/WESAD\data
Processing file: S2.pkl in G:/Master/Thesis/FLT/Code/WESAD\data
Completed 100/1000 runs
Completed 200/1000 runs
Completed 300/1000 runs
Completed 400/1000 runs
Completed 500/1000 runs
Completed 600/1000 runs
Completed 700/1000 runs
Completed 800/1000 runs
Completed 900/1000 runs
Completed 1000/1000 runs
